In [ ]:
# =============================================================================
#  Differential Expression Analysis (pseudobulk and single‑cell)
# =============================================================================

library(Seurat)
library(muscat)
library(limma)
library(dplyr)

# -----------------------------------------------------------------------------
# Load the integrated and annotated Seurat object
# -----------------------------------------------------------------------------
load('integrated_sc.RData')

# -----------------------------------------------------------------------------
# Pseudobulk differential expression using muscat (sex comparison)
# -----------------------------------------------------------------------------
# Switch to RNA assay for counts
DefaultAssay(combined) <- "RNA"

# Convert to SingleCellExperiment
pseudo <- as.SingleCellExperiment(combined)

# Filter genes: keep genes expressed in at least 10 cells with count >1
pseudo <- pseudo[rowSums(counts(pseudo) > 0) > 0, ]
pseudo <- pseudo[rowSums(counts(pseudo) > 1) >= 10, ]

# Prepare SCE for muscat: specify cluster id (celltype), group id (sex), sample id (library)
pseudo_prep <- prepSCE(pseudo,
                       kid = "celltype",      # cluster/cell type
                       gid = "sex",           # group of interest (e.g., "F", "M")
                       sid = "library",       # sample ID
                       drop = FALSE)

# Aggregate counts per cluster and sample
sex_pb <- aggregateData(pseudo_prep,
                        assay = "counts", fun = "sum",
                        by = c("cluster_id", "sample_id"))

# Extract experiment info and build design matrix
ei <- metadata(sex_pb)$experiment_info
design <- model.matrix(~ 0 + ei$group_id)
dimnames(design) <- list(ei$sample_id, levels(ei$group_id))

# Contrast: Female vs Male
contrast <- makeContrasts(F - M, levels = design)

# Run pseudobulk DE analysis with edgeR
sex_res <- pbDS(sex_pb, design = design, contrast = contrast,
                method = "edgeR", filter = 'both', min_cells = 10)

# Extract results per cluster and save as CSV
tbl <- sex_res$table[[1]]
for (j in 1:length(tbl)) {
  cluster_name <- names(tbl)[j]
  k1 <- tbl[[j]]
  print(paste(cluster_name, ':', nrow(subset(k1, p_adj.loc < 0.05)), sep = ''))
  write.csv(k1, paste0('muscat_deg/', cluster_name, '_sex.csv'))
}

# -----------------------------------------------------------------------------
# Single‑cell differential expression using Seurat (for each cell type)
# -----------------------------------------------------------------------------
# Create output directory if needed (uncomment if directory doesn't exist)
# dir.create('deg_seurat', showWarnings = FALSE)

for (cell in unique(combined$celltype)) {
  # Subset to current cell type
  data <- subset(combined, celltype == cell)
  
  # Sex comparison (F vs M)
  sex_deg <- FindMarkers(data,
                         ident.1 = 'F',
                         ident.2 = 'M',
                         group.by = 'sex',
                         logfc.threshold = 0.25,
                         pseudocount.use = 0.5)
  
  # Age comparison (old vs young)
  age_deg <- FindMarkers(data,
                         ident.1 = 'old',
                         ident.2 = 'young',
                         group.by = 'age_bin',
                         logfc.threshold = 0.25,
                         pseudocount.use = 0.5)
  
  # Phenotype comparison (AD vs non-AD)
  ad_deg <- FindMarkers(data,
                        ident.1 = 'AD',
                        ident.2 = 'non-AD',
                        group.by = 'pheno',
                        logfc.threshold = 0.25,
                        pseudocount.use = 0.5)
  
  # Save results
  write.csv(sex_deg, paste0('deg_seurat/', cell, '_sex.csv'))
  write.csv(age_deg, paste0('deg_seurat/', cell, '_age.csv'))
  write.csv(ad_deg,  paste0('deg_seurat/', cell, '_ad.csv'))
}


# Heatmap of Significant DEGs per Cell Type (Overall)

library(dplyr)
library(ggplot2)
library(tidyr)
library(stringr)
library(tools)

# Parameters
padj_cutoff <- 0.05
logfc_cutoff <- 0.25
target_celltypes <- c("EXC", "INH", "OLG", "OPC", "AST", "ENDO", "PER", "FIB", "MIC", "MAC", "TC")
group_order <- c("Sex", "Age", "AD")

# Read all CSV files in working directory
files <- list.files(pattern = "\\.csv$")
plot_list <- list()

for (f in files) {
  df <- read.csv(f)
  # Handle column name variations
  if ("p_val_adj" %in% colnames(df)) df <- rename(df, p_adj.loc = p_val_adj)
  if ("avg_log2FC" %in% colnames(df)) df <- rename(df, logFC = avg_log2FC)
  
  if (all(c("p_adj.loc", "logFC") %in% colnames(df))) {
    sig_count <- df %>% filter(p_adj.loc < padj_cutoff & abs(logFC) > logfc_cutoff) %>% nrow()
    
    # Parse filename: e.g., "AST_ad.csv" -> celltype = "AST", group = "AD"
    base <- file_path_sans_ext(basename(f))
    parts <- str_split(base, "_")[[1]]
    celltype <- paste(parts[1:(length(parts)-1)], collapse = "_")
    group <- parts[length(parts)]
    
    # Map to standard group names
    group <- case_when(group == "ad" ~ "AD", group == "disease" ~ "AD",
                       group == "age" ~ "Age", group == "sex" ~ "Sex",
                       TRUE ~ group)
    
    plot_list[[f]] <- data.frame(CellType = celltype, Group = group, Count = sig_count)
  }
}

# Combine and clean
heatmap_data <- bind_rows(plot_list) %>%
  mutate(CellType = case_when(CellType == "EC" ~ "ENDO", CellType == "INC" ~ "INH",
                              CellType == "Fibroblast" ~ "FIB", CellType == "Pericyte" ~ "PER",
                              CellType == "Macrophage" ~ "MAC", CellType == "T" ~ "TC",
                              TRUE ~ CellType)) %>%
  filter(CellType %in% target_celltypes) %>%
  complete(CellType, Group, fill = list(Count = 0)) %>%
  mutate(CellType = factor(CellType, levels = rev(target_celltypes)),
         Group = factor(Group, levels = group_order))

# Heatmap with numbers
p <- ggplot(heatmap_data, aes(x = Group, y = CellType, fill = Count)) +
  geom_tile(color = "white") +
  geom_text(aes(label = Count, color = Count > 500), size = 3.5, show.legend = FALSE) +
  scale_color_manual(values = c("TRUE" = "white", "FALSE" = "black")) +
  scale_fill_gradientn(colors = c("#FFF5F0", "#FEE0D2", "#FC9272", "#DE2D26", "#67000D"),
                       limits = c(0, 1000), oob = scales::squish, name = "# significant genes") +
  theme_minimal() + theme(axis.title = element_blank(), axis.text.x = element_text(angle = 0))
print(p)

# Heatmap of Significant DEGs by Brain Region (EXC and INH only)

# Similar to above but adds Region dimension
regions <- c("PFC", "SFG", "OC", "OTC", "M1", "MTG")
celltype_mapping <- c("EXC" = "EXC", "INC" = "INH", "EC" = "ENDO", "Fibroblast" = "FIB",
                      "Pericyte" = "PER", "Macrophage" = "MAC", "T" = "TC")
display_celltypes <- c("EXC", "INH")

all_data <- list()
for (region in regions) {
  region_path <- file.path("path/to/deg/folders", paste0(region, "_deg_seurat"))
  files <- list.files(region_path, pattern = "\\.csv$", full.names = TRUE)
  files <- files[grepl("^(EXC_|INC_)", basename(files))]
  
  for (f in files) {
    df <- read.csv(f)
    df <- rename(df, p_adj.loc = p_val_adj, logFC = avg_log2FC)  # adjust if needed
    sig_count <- df %>% filter(p_adj.loc < 0.05 & abs(logFC) > 0.25) %>% nrow()
    
    base <- file_path_sans_ext(basename(f))
    parts <- str_split(base, "_")[[1]]
    celltype_raw <- parts[1]
    celltype <- celltype_mapping[celltype_raw]
    if (is.na(celltype) || !celltype %in% display_celltypes) next
    
    all_data[[paste(region, celltype)]] <- data.frame(Region = region, CellType = celltype, Count = sig_count)
  }
}

heatmap_region <- bind_rows(all_data) %>%
  complete(CellType, Region, fill = list(Count = 0)) %>%
  mutate(CellType = factor(CellType, levels = rev(display_celltypes)),
         Region = factor(Region, levels = regions))

p <- ggplot(heatmap_region, aes(x = Region, y = CellType, fill = Count)) +
  geom_tile(color = "white") +
  geom_text(aes(label = Count), size = 3.5) +
  scale_fill_gradientn(colors = c("#FFF5F0", "#FEE0D2", "#FC9272", "#DE2D26", "#67000D"),
                       limits = c(0, max(heatmap_region$Count))) +
  theme_minimal() + theme(axis.title = element_blank())
print(p)

In [ ]:
# PART 3: AD Senescent Cell Differential Expression Analysis (originally fig3)
# ============================================================================
cat("\n========== PART 3: AD Senescent Cell DEG Analysis ==========\n")

# --- USER: Update these paths to your DEG CSV files ---
deg_file_list <- list(
    "AST" = "path/to/AD_AST_deg.csv",
    "MIC" = "path/to/AD_MIC_deg.csv",
    "PER" = "path/to/AD_PER_deg.csv",
    "TC"  = "path/to/AD_TC_deg.csv"
)

y_dash <- 0.05
log2fc_cutoff <- 0.25
top_n_per_group <- 10
output_dir <- "AD_volcano_plots"
if (!dir.exists(output_dir)) dir.create(output_dir, recursive = TRUE)

plot_volcano <- function(celltype, file_path) {
    deg_data <- read.csv(file_path, header = TRUE, stringsAsFactors = FALSE) %>%
        rename(gene = X, log2FC = avg_log2FC, p_adjust = p_val_adj) %>%
        mutate(
            min_p_nonzero = min(p_adjust[p_adjust > 0], na.rm = TRUE),
            log_p_val_adj = case_when(
                p_adjust == 0 ~ -log10(min_p_nonzero / 10),
                TRUE ~ -log10(p_adjust)
            ),
            change_type = case_when(
                log2FC > log2fc_cutoff ~ "up",
                log2FC < -log2fc_cutoff ~ "down",
                TRUE ~ "no_change"
            )
        ) %>%
        filter(!is.na(log2FC) & !is.na(p_adjust))
    
    top_up <- deg_data %>%
        filter(change_type == "up") %>%
        arrange(desc(log_p_val_adj)) %>%
        slice_head(n = top_n_per_group)
    top_down <- deg_data %>%
        filter(change_type == "down") %>%
        arrange(desc(log_p_val_adj)) %>%
        slice_head(n = top_n_per_group)
    top_genes <- bind_rows(top_up, top_down)
    
    y_max <- max(deg_data$log_p_val_adj, na.rm = TRUE) * 1.2
    
    p <- ggplot(deg_data, aes(x = log2FC, y = log_p_val_adj)) +
        geom_point(aes(size = log_p_val_adj, color = log_p_val_adj), alpha = 0.8) +
        scale_color_gradientn(colors = c("lightblue", "yellow", "red")) +
        scale_size_continuous(range = c(0.1, 6)) +
        geom_hline(yintercept = y_dash, linetype = "dashed", color = "black", linewidth = 1) +
        geom_vline(xintercept = c(-log2fc_cutoff, log2fc_cutoff),
                   linetype = "dashed", color = "black", linewidth = 1) +
        scale_x_continuous(limits = c(-1.5, 1.5), breaks = seq(-1.5, 1.5, 0.5)) +
        scale_y_continuous(limits = c(0, y_max), expand = c(0, 0)) +
        labs(
            x = "log2 Fold Change",
            y = "-log10(adjusted p-value)",
            title = paste("AD", celltype, "Differential Expression Genes")
        ) +
        theme_minimal() +
        theme(
            text = element_text(size = 16),
            plot.title = element_text(hjust = 0.5, size = 18, face = "bold")
        ) +
        guides(size = "none")
    
    if (nrow(top_genes) > 0) {
        p <- p + ggrepel::geom_text_repel(
            data = top_genes,
            aes(label = gene),
            size = 3.5,
            fontface = "bold",
            box.padding = 0.5,
            max.overlaps = 200
        )
    }
    
    out_file <- file.path(output_dir, paste0("AD_", celltype, "_volcano.pdf"))
    ggsave(out_file, p, width = 10, height = 8, dpi = 300)
    cat(celltype, "volcano plot saved.\n")
}

for (ct in names(deg_file_list)) {
    if (file.exists(deg_file_list[[ct]])) {
        plot_volcano(ct, deg_file_list[[ct]])
    } else {
        warning("File not found: ", deg_file_list[[ct]])
    }
}

# -------- 3.2 Bar plot of up/down DEG counts (E) ----------
all_degs <- data.frame()
for (ct in names(deg_file_list)) {
    if (file.exists(deg_file_list[[ct]])) {
        deg <- read.csv(deg_file_list[[ct]], header = TRUE, stringsAsFactors = FALSE) %>%
            mutate(
                celltype = ct,
                avg_log2FC = ifelse("avg_log2FC" %in% names(.), avg_log2FC, NA),
                p_val_adj = ifelse("p_val_adj" %in% names(.), p_val_adj, NA)
            ) %>%
            filter(!is.na(avg_log2FC) & !is.na(p_val_adj))
        all_degs <- bind_rows(all_degs, deg)
    }
}

deg_counts <- all_degs %>%
    mutate(
        direction = case_when(
            avg_log2FC > 0.25 ~ "Upregulated",
            avg_log2FC < -0.25 ~ "Downregulated",
            TRUE ~ "Not significant"
        )
    ) %>%
    filter(p_val_adj < 0.05, direction != "Not significant") %>%
    group_by(celltype, direction) %>%
    summarise(count = n(), .groups = "drop")

p_bar <- ggplot(deg_counts, aes(x = celltype, y = count, fill = direction)) +
    geom_bar(stat = "identity", position = position_dodge(width = 0.8), width = 0.7) +
    scale_fill_manual(values = c("Upregulated" = "red", "Downregulated" = "blue")) +
    labs(
        title = "Number of Up/Down Regulated DEGs in AD Cell Types",
        x = "Cell Type",
        y = "Number of Significant DEGs",
        fill = "Regulation Direction"
    ) +
    theme_classic() +
    theme(
        plot.title = element_text(hjust = 0.5, size = 16, face = "bold"),
        axis.text.x = element_text(size = 14, face = "bold"),
        axis.text.y = element_text(size = 12),
        axis.title = element_text(size = 14, face = "bold"),
        legend.title = element_text(size = 12, face = "bold"),
        legend.text = element_text(size = 10)
    ) +
    geom_text(
        aes(label = count),
        position = position_dodge(width = 0.8),
        vjust = -0.3,
        size = 4,
        fontface = "bold"
    )

ggsave("AD_celltype_deg_counts_barplot.pdf", p_bar, width = 10, height = 7, dpi = 300)
cat("Bar plot of DEG counts saved.\n")

# -------- 3.3 Faceted violin plots for selected genes (H) ----------
genes_H <- c(
    "IL6ST", "STAT1", "TGFB1", "B2M", "C3", "SPP1",
    "ANGPT1", "ANGPTL4", "FGF1", "FGF2", "GFAP", "CD44",
    "ADGRB3", "DAB1", "CDH2", "CDH10", "CD247",
    "LCK", "RORA", "STAT4", "FYN"
)
split_cell_types_H <- c("AST", "PER", "MIC", "TC")
sample_n <- 500

meta_H <- as.data.frame(seurat_obj@meta.data)
meta_H$cell <- rownames(meta_H)
meta_H <- meta_H[meta_H[[cell_type_col]] %in% split_cell_types_H, ]
meta_H$ageing_cell <- ifelse(
    meta_H[[senescence_col]] == "Non-senescent",
    "normal", "ageing_cell"
)
meta_H$group <- paste0(meta_H[[cell_type_col]], "_", meta_H$ageing_cell)
meta_H <- meta_H[, c("cell", "group")]

set.seed(123)
sampled_cells <- unlist(lapply(unique(meta_H$group), function(g) {
    cells <- meta_H$cell[meta_H$group == g]
    sample(cells, size = min(length(cells), sample_n), replace = FALSE)
}))

valid_genes_H <- intersect(genes_H, rownames(seurat_obj))
expr_matrix <- as.matrix(GetAssayData(seurat_obj, slot = "data")[valid_genes_H, sampled_cells])
expr_long <- as.data.frame(t(expr_matrix)) %>%
    mutate(cell = rownames(.)) %>%
    pivot_longer(cols = -cell, names_to = "gene", values_to = "expression")

final_data <- merge(expr_long, meta_H[meta_H$cell %in% sampled_cells, ], by = "cell")

custom_order_H <- paste0(rep(split_cell_types_H, each = 2), "_", c("normal", "ageing_cell"))
final_data$group <- factor(final_data$group, levels = custom_order_H)
final_data$gene <- factor(final_data$gene, levels = valid_genes_H)

cell_colors_H <- c(
    "AST_normal"       = "#FF9999", "AST_ageing_cell" = "#CC0000",
    "PER_normal"       = "#FFDAB9", "PER_ageing_cell" = "#CD853F",
    "MIC_normal"       = "#99FF99", "MIC_ageing_cell" = "#009933",
    "TC_normal"        = "#E6E6FA", "TC_ageing_cell"  = "#483D8B"
)

p_facet <- ggplot(final_data, aes(x = group, y = expression, fill = group)) +
    geom_violin(trim = TRUE, scale = "area", na.rm = TRUE) +
    facet_grid(rows = vars(gene), scales = "free_y", switch = "y") +
    scale_fill_manual(values = cell_colors_H, drop = FALSE) +
    scale_x_discrete(drop = FALSE) +
    theme_minimal() +
    theme(
        strip.text.y.left = element_text(angle = 0, hjust = 0, size = 10, face = "bold"),
        strip.placement = "outside",
        axis.text.x = element_text(angle = 60, hjust = 1, size = 8),
        axis.title.x = element_blank(),
        legend.position = "none",
        panel.spacing = unit(0.2, "lines")
    ) +
    labs(y = NULL)

ggsave("gene_violin_final.pdf", p_facet, width = 20, height = 28, dpi = 300, bg = "white")
cat("Faceted violin plot saved.\n")

cat("\n========== All analyses completed ==========\n")